# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

In [ ]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)


## 🌟 Exercise 1 · Data loading and preparation

In [ ]:
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))  # replace with your own identifier logic if provided
display(pdf.head())
# create a manageable subset (e.g., first 1000 rows)
pdf_subset = pdf.head(1000)
pdf_subset[['id', 'title']].head()


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [3]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# create training examples from the subset data using the example_create_fn
faiss_train_examples = [example_create_fn(idx, text) for idx, text in pdf_subset[['id', 'title']].values]
faiss_train_examples[:2]

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])


## 🌟 Exercise 3 · FAISS indexing and search

In [5]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


1000

In [7]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # Encode the query using the sentence transformer model
    query_vector = model.encode(query, convert_to_numpy=True, show_progress_bar=False)

    # Reshape to 2D array (1 query, vector_dimension)
    query_vector = query_vector.reshape(1, -1)

    # Normalize the query vector
    faiss.normalize_L2(query_vector)

    # Search for similar vectors in the index
    sims, ids = index_content.search(query_vector, k)

    # Get results from the original dataframe
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]

    # Sort by similarity score (descending order)
    results = results.sort_values('similarities', ascending=False)

    return results

# Test the function
display(search_content('animal', pdf_to_index, k=5))

,topic,link,domain,published_date,title,lang,id,similarities
99,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en,99,0.391902
176,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en,176,0.376784
762,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en,762,0.344059
928,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en,928,0.317387
975,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en,975,0.295497


## 🌟 Exercise 4 · ChromaDB collection and querying

In [ ]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)
collection.add(
    documents=pdf_to_index['title'].tolist(),
    metadatas=[{}] * len(pdf_to_index),
    ids=pdf_to_index['id'].astype(str).tolist()
)
results = collection.query(
    query_texts=["animal"],
    n_results=5
)
print(json.dumps(results, indent=2))

## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [21]:
model_id = 'google/flan-t5-small'
pipe = pipeline("text2text-generation", model=model_id)

question = 'Which titles include the name of a video game?'
context_docs = results['documents'][0][:3]
context = ' '.join(context_docs)
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"
response = pipe(prompt)[0]['generated_text']
print(response)

Device set to use cpu


Random: You Can Pick Up and Pet Cats in Assassin's Creed Valhalla
